In [ ]:
import zipfile

zip_path = "datasets.zip"
extract_path = "."

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped!")



Unzipped!


In [ ]:

#we are using the networkx library to represent our graph
import re
import networkx as nx
from collections import deque

KB_PATH = "datasets/kb.txt"

QA_1HOP_TEST = "datasets/1-hop/qa_test.txt"
QA_2HOP_TEST = "datasets/2-hop/qa_test.txt"
QA_3HOP_TEST = "datasets/3-hop/qa_test.txt"

print("Imports OK")



Imports OK


In [ ]:
#just loading the graph from the triples, only thing we add is the inverse edges (ex: starred_in becomes inv_starred_in in the other direction)
def load_kb(path, add_inverse_edges=True):
    G = nx.DiGraph()

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            parts = line.split("|")
            if len(parts) != 3:
                continue

            head, rel, tail = [p.strip() for p in parts]

            G.add_edge(head, tail, relation=rel)

            if add_inverse_edges:
                G.add_edge(tail, head, relation=f"inv_{rel}")

    return G


In [ ]:
#just a parser for the qa dataset lines
def load_qa(path):
    examples = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            parts = line.split("\t")
            if len(parts) != 2:
                continue

            question, answer_text = parts
            answers = [a.strip() for a in answer_text.split("|")]

            examples.append({
                "question": question,
                "answers": answers,
                "topic_entity": extract_topic_entity(question),
            })

    return examples


In [ ]:

#using regex to see what the main entity in the question is, this'll become the starting node for the search
def extract_topic_entity(question):
    match = re.search(r"\[(.*?)\]", question)

    if match:
        return match.group(1).strip()

    return None



In [ ]:
#this is just retrieving all the neighbors until max_hops, pretty much just bfs
def get_k_hop_candidates(G, start_entity, max_hops):

    if start_entity not in G:
        return []

    queue = deque()
    queue.append((start_entity, 0, [start_entity]))

    results = []
    visited = set([(start_entity, 0)])

    while queue:
        node, depth, path = queue.popleft()

        if depth == max_hops:
            results.append({
                "candidate": node,
                "path": path,
            })
            continue

        for neighbor in G.neighbors(node):
            state = (neighbor, depth + 1)

            if state in visited:
                continue

            visited.add(state)
            queue.append((neighbor, depth + 1, path + [neighbor]))

    return results



In [ ]:
#just verifying if traversal is working correctly
def print_sample_run(G, qa_path, hops):
    examples = load_qa(qa_path)

    sample = examples[0]
    question = sample["question"]
    answers = sample["answers"]
    topic_entity = sample["topic_entity"]

    print("\nSample")
    print("Question:", question)
    print("Topic entity:", topic_entity)
    print("Gold answers:", answers)

    candidates = get_k_hop_candidates(G, topic_entity, hops)

    candidate_names = [x["candidate"] for x in candidates]
    hits = sorted(set(candidate_names).intersection(set(answers)))

    print(f"{hops}-hop candidate count:", len(candidate_names))
    print("Hits:", hits)
    print("First 20 candidates:", candidate_names[:20])


if __name__ == "__main__":
    print("Loading KB...")
    G = load_kb(KB_PATH, add_inverse_edges=True)

    print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} directed edges")

    print_sample_run(G, QA_1HOP_TEST, hops=1)
    print_sample_run(G, QA_2HOP_TEST, hops=2)
    print_sample_run(G, QA_3HOP_TEST, hops=3)

Loading KB...
Graph loaded: 43234 nodes, 249349 directed edges

Sample
Question: what does [Grégoire Colin] appear in
Topic entity: Grégoire Colin
Gold answers: ['Before the Rain']
1-hop candidate count: 1
Hits: ['Before the Rain']
First 20 candidates: ['Before the Rain']

Sample
Question: which person directed the movies starred by [John Krasinski]
Topic entity: John Krasinski
Gold answers: ['Nancy Meyers', 'Sam Mendes', 'George Clooney', 'Ken Kwapis', 'Luke Greenfield']
2-hop candidate count: 59
Hits: ['George Clooney', 'Ken Kwapis', 'Luke Greenfield', 'Nancy Meyers', 'Sam Mendes']
First 20 candidates: ['George Clooney', 'Rick Reilly', 'Duncan Brantley', 'John Krasinski', '2008', 'Comedy', 'comedy', 'sports', 'george clooney', 'football', 'jonathan pryce', 'Sam Mendes', 'Dave Eggers', 'Vendela Vida', "Catherine O'Hara", 'Maya Rudolph', 'Carmen Ejogo', '2009', 'Drama', 'r']

Sample
Question: the films that share directors with the film [Catch Me If You Can] were in which languages
Top

In [ ]:
#making sure that the dataset is fully loaded, and checking dataset statistics
#the important thing we do here for trans-E is that we basically assign ids to each relation type
#and each entity
#then we save it to a json file, to use it later with pykeen
from pathlib import Path
import random
import json

KB_PATH = Path("datasets/kb.txt")
OUT_DIR = Path("transe_data")
OUT_DIR.mkdir(exist_ok=True)

ADD_INVERSE = True
SEED = 42

random.seed(SEED)

entities = {}
relations = {}
triples = []


def get_id(mapping, key):
    if key not in mapping:
        mapping[key] = len(mapping)
    return mapping[key]


with open(KB_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        h, r, t = [x.strip() for x in line.split("|")]

        h_id = get_id(entities, h)
        r_id = get_id(relations, r)
        t_id = get_id(entities, t)

        triples.append((h_id, r_id, t_id))

        if ADD_INVERSE:
            inv_r = f"inv_{r}"
            inv_r_id = get_id(relations, inv_r)
            triples.append((t_id, inv_r_id, h_id))


random.shuffle(triples)

n = len(triples)
train = triples[: int(0.9 * n)]
valid = triples[int(0.9 * n): int(0.95 * n)]
test = triples[int(0.95 * n):]


def save_triples(path, data):
    with open(path, "w", encoding="utf-8") as f:
        for h, r, t in data:
            f.write(f"{h}\t{r}\t{t}\n")


save_triples(OUT_DIR / "train.txt", train)
save_triples(OUT_DIR / "valid.txt", valid)
save_triples(OUT_DIR / "test.txt", test)

with open(OUT_DIR / "entity2id.json", "w", encoding="utf-8") as f:
    json.dump(entities, f, ensure_ascii=False, indent=2)

with open(OUT_DIR / "relation2id.json", "w", encoding="utf-8") as f:
    json.dump(relations, f, ensure_ascii=False, indent=2)

print("Done")
print("Entities:", len(entities))
print("Relations:", len(relations))
print("Train triples:", len(train))
print("Valid triples:", len(valid))
print("Test triples:", len(test))

Done
Entities: 43234
Relations: 18
Train triples: 242533
Valid triples: 13474
Test triples: 13475


In [ ]:
#load the json files we just saved with the mappings from relation to relation id and entity to entity id
#then we just make train/test/validation splits to use with the model
import json
from pathlib import Path

DATA_DIR = Path("transe_data")

with open(DATA_DIR / "entity2id.json") as f:
    entity2id = json.load(f)

with open(DATA_DIR / "relation2id.json") as f:
    relation2id = json.load(f)

id2entity = {v: k for k, v in entity2id.items()}
id2relation = {v: k for k, v in relation2id.items()}


def convert_file(in_path, out_path):
    with open(in_path, "r") as fin, open(out_path, "w") as fout:
        for line in fin:
            h, r, t = map(int, line.strip().split("\t"))

            h_str = id2entity[h]
            r_str = id2relation[r]
            t_str = id2entity[t]

            fout.write(f"{h_str}\t{r_str}\t{t_str}\n")


#just getting spplits for training
convert_file(DATA_DIR / "train.txt", DATA_DIR / "train_str.txt")
convert_file(DATA_DIR / "valid.txt", DATA_DIR / "valid_str.txt")
convert_file(DATA_DIR / "test.txt", DATA_DIR / "test_str.txt")

print("Converted to string format!")

Converted to string format!


In [ ]:
#passing the full training set to pykeen to learn the embeddings
# using CUDA/GPU because we ran this on google colab
# might run for a very long time on CPU
import torch
from pathlib import Path
from pykeen.pipeline import pipeline

print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

# This is just to speed up evaluation, we are using a small subset of the testset
tiny_test_path = Path("transe_data/tiny_test_str.txt")

with open("transe_data/test_str.txt", "r", encoding="utf-8") as fin, \
     open(tiny_test_path, "w", encoding="utf-8") as fout:
    for i, line in enumerate(fin):
        if i >= 10:
            break
        fout.write(line)

result = pipeline(
    training="transe_data/train_str.txt",
    testing=str(tiny_test_path),
    validation=None,
    model="TransE",
    model_kwargs=dict(
        embedding_dim=100,
        scoring_fct_norm=1,
    ),
    training_kwargs=dict(
        num_epochs=100,
        batch_size=4096,
    ),
    optimizer="Adam",
    optimizer_kwargs=dict(
        lr=0.001,
    ),
    random_seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

result.save_to_directory("transe_model")

print("Training complete and model saved!")


CUDA available: True
Device: cuda


INFO:pykeen.pipeline.api:Using device: cuda
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()


Training epochs on cuda:0:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/59.0 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/59.0 [00:00<?, ?batch/s]

...

Training batches on cuda:0:   0%|          | 0.00/59.0 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/59.0 [00:00<?, ?batch/s]

Evaluating on cuda:0:   0%|          | 0.00/10.0 [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 0.80s seconds
INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=43102, num_relations=18, create_inverse_triples=False, num_triples=240618, path="/content/transe_data/train_str.txt") to file:///content/transe_model/training_triples
INFO:pykeen.pipeline.api:Saved to directory: /content/transe_model


Training complete and model saved!


In [ ]:
#this is the core of the algorithm
# we just generate the embeddings for all the neighbors we explore
# then we just use the score_triple function from pykeen which essentially checks the distance || h + r - t ||
def transe_beam_immediate_prune(G, start_entity, max_hops, beam_size):
    beam = [{
        "node": start_entity,
        "path_len": 0,
        "score": 0.0,
    }]

    total_edges_checked = 0
    total_edges_scored = 0

    for hop in range(max_hops):
        expanded = []

        for item in beam:
            node = item["node"]

            scored_neighbors = []

            for neighbor in G.neighbors(node):
                total_edges_checked += 1
                rel = G[node][neighbor]["relation"]

                s = score_triple(node, rel, neighbor)
                if s is None:
                    continue

                total_edges_scored += 1

                new_score = (
                    item["score"] * item["path_len"] + s
                ) / (item["path_len"] + 1)

                scored_neighbors.append({
                    "node": neighbor,
                    "path_len": item["path_len"] + 1,
                    "score": new_score,
                })
            scored_neighbors.sort(key=lambda x: x["score"], reverse=True)
            expanded.extend(scored_neighbors[:beam_size])
        expanded.sort(key=lambda x: x["score"], reverse=True)
        beam = expanded[:beam_size]

        if not beam:
            break

    final_nodes = set(x["node"] for x in beam)

    return final_nodes, total_edges_checked, total_edges_scored

#then we run bfs, and at each step prune based on the beam size (we try 5 - 250)
# and then check how much edge reduction we achieve (just comparing what percent of bfs edges remain in the pruned search)
# also track recall --> in the pruned graph, how often are the gold candidates still present6
def evaluate_file_immediate_pruning(
    qa_path,
    hops,
    beam_sizes=(5, 10, 25, 50, 100, 250),
    limit=200,
):
    examples = load_qa(qa_path)

    if limit is not None:
        examples = examples[:limit]

    print("=" * 90)
    print(f"Evaluating {qa_path} | hops={hops} | examples={len(examples)}")

    bfs_hits = 0
    bfs_edges_total = 0
    bfs_candidates_total = 0

    beam_stats = {
        b: {
            "hits": 0,
            "edges_checked": 0,
            "edges_scored": 0,
            "candidates": 0,
        }
        for b in beam_sizes
    }

    skipped = 0

    for ex in tqdm(examples):
        topic = ex["topic_entity"]
        gold = ex["answers"]

        if topic not in G:
            skipped += 1
            continue

        bfs_candidates, bfs_edges = raw_bfs_final(G, topic, hops)
        bfs_edges_total += bfs_edges
        bfs_candidates_total += len(bfs_candidates)

        if bfs_candidates.intersection(gold):
            bfs_hits += 1

        for b in beam_sizes:
            candidates, edges_checked, edges_scored = transe_beam_immediate_prune(
                G,
                topic,
                max_hops=hops,
                beam_size=b,
            )

            beam_stats[b]["edges_checked"] += edges_checked
            beam_stats[b]["edges_scored"] += edges_scored
            beam_stats[b]["candidates"] += len(candidates)

            if candidates.intersection(gold):
                beam_stats[b]["hits"] += 1

    n = len(examples) - skipped

    print("\nRaw BFS")
    print("valid examples:", n)
    print("recall:", bfs_hits / max(n, 1))
    print("avg edges checked:", bfs_edges_total / max(n, 1))
    print("avg final candidates:", bfs_candidates_total / max(n, 1))

    print("\nImmediate TransE pruning")
    for b in beam_sizes:
        s = beam_stats[b]

        print(
            f"beam={b:<4} "
            f"recall={s['hits'] / max(n, 1):.4f} | "
            f"avg_edges_checked={s['edges_checked'] / max(n, 1):.1f} | "
            f"avg_edges_scored={s['edges_scored'] / max(n, 1):.1f} | "
            f"avg_final_candidates={s['candidates'] / max(n, 1):.1f} | "
            f"edge_reduction={1 - (s['edges_checked'] / max(bfs_edges_total, 1)):.2%}"
        )


evaluate_file_immediate_pruning("datasets/1-hop/qa_test.txt", hops=1, limit=200)
evaluate_file_immediate_pruning("datasets/2-hop/qa_test.txt", hops=2, limit=200)
evaluate_file_immediate_pruning("datasets/3-hop/qa_test.txt", hops=3, limit=200)

Evaluating datasets/1-hop/qa_test.txt | hops=1 | examples=200


100%|██████████| 200/200 [00:01<00:00, 130.46it/s]



Raw BFS
valid examples: 200
recall: 1.0
avg edges checked: 3.165
avg final candidates: 3.165

Immediate TransE pruning
beam=5    recall=0.9750 | avg_edges_checked=3.2 | avg_edges_scored=3.2 | avg_final_candidates=2.2 | edge_reduction=0.00%
beam=10   recall=0.9800 | avg_edges_checked=3.2 | avg_edges_scored=3.2 | avg_final_candidates=2.7 | edge_reduction=0.00%
beam=25   recall=0.9900 | avg_edges_checked=3.2 | avg_edges_scored=3.2 | avg_final_candidates=3.1 | edge_reduction=0.00%
beam=50   recall=0.9900 | avg_edges_checked=3.2 | avg_edges_scored=3.2 | avg_final_candidates=3.2 | edge_reduction=0.00%
beam=100  recall=0.9900 | avg_edges_checked=3.2 | avg_edges_scored=3.2 | avg_final_candidates=3.2 | edge_reduction=0.00%
beam=250  recall=0.9900 | avg_edges_checked=3.2 | avg_edges_scored=3.2 | avg_final_candidates=3.2 | edge_reduction=0.00%
Evaluating datasets/2-hop/qa_test.txt | hops=2 | examples=200


100%|██████████| 200/200 [04:34<00:00,  1.37s/it]



Raw BFS
valid examples: 200
recall: 1.0
avg edges checked: 612.3
avg final candidates: 567.805

Immediate TransE pruning
beam=5    recall=0.6250 | avg_edges_checked=244.8 | avg_edges_scored=244.8 | avg_final_candidates=4.7 | edge_reduction=60.03%
beam=10   recall=0.7800 | avg_edges_checked=508.8 | avg_edges_scored=508.8 | avg_final_candidates=8.6 | edge_reduction=16.90%
beam=25   recall=0.9000 | avg_edges_checked=611.6 | avg_edges_scored=611.6 | avg_final_candidates=16.5 | edge_reduction=0.11%
beam=50   recall=0.9400 | avg_edges_checked=612.3 | avg_edges_scored=612.3 | avg_final_candidates=25.5 | edge_reduction=0.00%
beam=100  recall=0.9500 | avg_edges_checked=612.3 | avg_edges_scored=612.3 | avg_final_candidates=40.1 | edge_reduction=0.00%
beam=250  recall=0.9800 | avg_edges_checked=612.3 | avg_edges_scored=612.3 | avg_final_candidates=75.1 | edge_reduction=0.00%
Evaluating datasets/3-hop/qa_test.txt | hops=3 | examples=200


100%|██████████| 200/200 [27:38<00:00,  8.29s/it]


Raw BFS
valid examples: 200
recall: 1.0
avg edges checked: 28844.645
avg final candidates: 8839.325

Immediate TransE pruning
beam=5    recall=0.3000 | avg_edges_checked=1383.0 | avg_edges_scored=1382.9 | avg_final_candidates=3.2 | edge_reduction=95.21%
beam=10   recall=0.3700 | avg_edges_checked=2423.3 | avg_edges_scored=2423.2 | avg_final_candidates=6.0 | edge_reduction=91.60%
beam=25   recall=0.4550 | avg_edges_checked=3284.4 | avg_edges_scored=3284.2 | avg_final_candidates=14.2 | edge_reduction=88.61%
beam=50   recall=0.6100 | avg_edges_checked=3486.8 | avg_edges_scored=3486.3 | avg_final_candidates=28.6 | edge_reduction=87.91%
beam=100  recall=0.7000 | avg_edges_checked=3873.3 | avg_edges_scored=3872.4 | avg_final_candidates=61.8 | edge_reduction=86.57%
beam=250  recall=0.8250 | avg_edges_checked=5129.8 | avg_edges_scored=5127.9 | avg_final_candidates=165.5 | edge_reduction=82.22%
